# Preamble

In [1]:
# Running this script on a AWS Cloud SageMaker Notebook (ml.m5.2xlarge) with a volume size of 50 GB EBS

!python --version

Python 3.10.20


In [2]:
# Install Python packages that don't come pre-installed in AWS

!pip install scikit-learn
!pip install statsmodels

In [3]:
# Load required libraries

import boto3
import pandas as pd
import numpy as np
import re

import statsmodels.api as sm

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Import Data

In [4]:
# Downloaded the CFPB compliants data from this website on June 16, 2026

# https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data

In [5]:
# Import the CSV file of complaints locally OR from the AWS S3 bucket

# https://aletheia-public.s3.us-east-2.amazonaws.com/complaints_16Jun2026.csv

load_data_locally = False

if load_data_locally:
    df = pd.read_csv("complaints_in_just_2025.csv")  # just data from 2025
else:
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket='aletheia-public', Key='complaints_16Jun2026.csv')  # all years of data
    df = pd.read_csv(obj['Body'])

/tmp/ipykernel_32665/105650091.py:12: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(obj['Body'])


In [6]:
# The full dataset of complaints has 15,896,496 observations

df.shape

(15896496, 16)

In [7]:
# Below is a data dictionary of all columns in the CFPB compliants data

# https://cfpb.github.io/api/ccdb/fields.html

In [8]:
# List the raw data types of all columns of the Pandas data frame

df.dtypes

Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

In [9]:
# Convert the date columns from a string object format to a datetime format with just a date (i.e. YYYY-MM-DD)

df['Date received'] = pd.to_datetime(df['Date received'].str[:10]).dt.normalize()
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'].str[:10]).dt.normalize()

In [10]:
# Look at a few obs

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,CL Holdings LLC,CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907454
1,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Portfolio Recovery Associates, LLC",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907265
2,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Portfolio Recovery Associates, LLC",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907334
3,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company believes it acted appropriately as aut...,"Diversified Adjustment Service, Inc.",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907328
4,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Adler Wallach & Associates, Inc.",CA,90061,NaN,Web,2023-04-30,Closed with explanation,No,6907322


In [11]:
# Group complaints by year received and count observations

df.groupby(df['Date received'].dt.year).size()

Date received
2011       2536
2012      72368
2013     108214
2014     152909
2015     168273
2016     191294
2017     242747
2018     257133
2019     277240
2020     444214
2021     495942
2022     800257
2023    1292053
2024    2734271
2025    5443005
2026    3214040
dtype: int64

In [14]:
# Sample the data over different years and save as a CSV file for Alex

if not load_data_locally:
    df_sampled = df.sample(frac=0.25, random_state=42)
    df_sampled.to_csv('complaints_16Jun2026_sampled.csv', index=False)

In [ ]:
# Filter complaints in just the year 2025 in order to reduce the size of the Pandas data frame

df = df[df['Date received'].dt.year == 2025]

In [ ]:
# Confirm the count of observations in year 2025

df.shape

# Explore Data

In [ ]:
# Group complaints by product category and count observations

df.groupby(df['Product']).size()

In [ ]:
# Convert the Product categorical variable into a collectino of dummy variables
# use the category 'Credit reporting or other personal consumer reports' as the baseline category (because it is the most common)

df['Product'] = pd.Categorical(df['Product'], categories=['Credit reporting or other personal consumer reports',
                                                          'Checking or savings account', 
                                                          'Credit card', 
                                                          'Debt collection', 
                                                          'Debt or credit management', 
                                                          'Money transfer, virtual currency, or money service', 
                                                          'Mortgage', 
                                                          'Payday loan, title loan, personal loan, or advance loan', 
                                                          'Prepaid card', 
                                                          'Student loan', 
                                                          'Vehicle loan or lease'])

df = pd.get_dummies(df, columns=['Product'], drop_first=True, dtype=int)

In [ ]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

In [ ]:
# Group by the tags and count obsevations

df.groupby(df['Tags']).size()

In [ ]:
# Recode this categorical feature of 'Tags' into two separate binary features

df['Older American'] = np.where(df['Tags'].isin(['Older American','Older American, Servicemember']), 1, 0)
df['Servicemember'] = np.where(df['Tags'].isin(['Servicemember','Older American, Servicemember']), 1, 0)

In [ ]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

In [ ]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

In [ ]:
# Collapse company response to consumer into a binary target: 

#   1 = complaint closed respectfully (with an explanation OR a monetary relief)
#   0 = complaint closed with no explanation

df['Target of Complaint Closed Respectfully'] = np.where(df['Company response to consumer'].isin(['Closed with explanation','Closed with monetary relief']), 1, 0)

In [ ]:
# Group by target and count obsevations

df.groupby(df['Target of Complaint Closed Respectfully']).size()

In [ ]:
# Group by timely response to consumer complaints and count obsevations

df.groupby(df['Timely response?']).size()

In [ ]:
# Group by how complaint was submitted via and count obsevations

df.groupby(df['Submitted via']).size()

In [ ]:
# Convert the Product categorical variable into a collection of dummy variables 
# Use the category 'Web' as the baseline category (because it is the most common)

df['Submitted via'] = pd.Categorical(df['Submitted via'], categories=['Web', 
                                                                      'Phone', 
                                                                      'Postal mail', 
                                                                      'Referral'])

df = pd.get_dummies(df, columns=['Submitted via'], drop_first=True, dtype=int)

In [ ]:
# Count up the number of unique companies of consumer complaints

df['Company'].nunique()

In [ ]:
# Find the top 15 companies with the most consumer complaints

df['Company'].value_counts().head(20)

In [ ]:
# Print consumer complaints where the complaint narrative contains a specific keyword

keyword = 'Data Science'

with pd.option_context('display.max_colwidth', None):
    filtered_column = df.loc[df['Consumer complaint narrative'].str.contains(keyword, case=False, na=False), 'Consumer complaint narrative']
    print(filtered_column)

In [ ]:
# Count up the number of actual complaint narratives: 1,221,970 out of 5,443,005 registered complaints in 2026

df[['Consumer complaint narrative']].count(axis=0)

# Feature Engineering

In [ ]:
# Print out columns in the data frame

print(df.columns.tolist())

In [ ]:
# Count up the words in the consumer complaint narrative

df['complaint_word_count'] = df['Consumer complaint narrative'].str.split().str.len()

In [ ]:
# Binary flag if the $ dollar sign in the consumer complaint narrative

df['dollar_sign'] = df['Consumer complaint narrative'].str.contains('$', na=False).astype(int)

In [ ]:
# Binary flag if there are keywords related to fraud in the consumer complaint narrative

fraud_keywords = ['fraud', 'fraudulent', 'fake', 'forgery', 'trick', 'scam', 'swindle', 'hoax', 'scheme', 'sham']
fraud_pattern = '|'.join(fraud_keywords)

df['fraud_keyword'] = df['Consumer complaint narrative'].str.contains(fraud_pattern, case=False, na=False).astype(int)

In [ ]:
# Binary flag if the company is a credit bureau 

df['credit_bureau'] = np.where(df['Company'].isin(['EQUIFAX, INC.',
                                                   'TRANSUNION INTERMEDIATE HOLDINGS, INC.', 
                                                   'Experian Information Solutions Inc.']), 1, 0)

In [ ]:
# Binary flag if the company is a credit union

df['credit_union'] = df['Company'].str.contains('credit union', case=False, na=False).astype(int)

In [ ]:
# Convert NaN to 0

df = df.fillna(0)

In [ ]:
df.shape

# Prepare for Modeling

In [ ]:
# Subset just the relevant columns in the data frame

relevant_columns = ['Target of Complaint Closed Respectfully',
                    'Product_Checking or savings account',
                    'Product_Credit card',
                    'Product_Debt collection',
                    'Product_Debt or credit management',
                    'Product_Money transfer, virtual currency, or money service',
                    'Product_Mortgage',
                    'Product_Payday loan, title loan, personal loan, or advance loan',
                    'Product_Prepaid card',
                    'Product_Student loan',
                    'Product_Vehicle loan or lease',
                    'Older American',
                    'Servicemember',
                    'Submitted via_Phone',
                    'Submitted via_Postal mail',
                    'Submitted via_Referral',
                    'complaint_word_count',
                    'dollar_sign',
                    'fraud_keyword',
                    'credit_bureau',
                    'credit_union']

subset_df = df[relevant_columns]

In [ ]:
# Look at the data subset

subset_df.head()

In [ ]:
# Separate feature vector (X) and target variable (y)

X = subset_df.drop('Target of Complaint Closed Respectfully', axis=1)
y = subset_df['Target of Complaint Closed Respectfully']

In [ ]:
# Split the data into a training partition and a hold-out validation partition (70% training, 30% holdout)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Logistic Regression Model using Statsmodels

In [ ]:
# Add a column of 1s to the feature set because statsmodels does not automatically add an intercept

X_train_with_constant = sm.add_constant(X_train)

In [ ]:
# Fit a logistic regression model using statsmodel on the training data with robust standard errors
# https://blog.stata.com/2022/10/06/heteroskedasticity-robust-standard-errors-some-practical-considerations/

logistic_reg_stats_models = sm.Logit(y_train, X_train_with_constant).fit(cov_type='HC1')

In [ ]:
# 2. View statistical summary of the results of fitting a Logistic regression model

print(logistic_reg_stats_models.summary())

In [ ]:
# Convert the Logistic regression model coefficients into Odds Ratios

# Extract log-odds coefficients and confidence intervals
df_coefficients = logistic_reg_stats_models.params.to_frame(name='Log-Odds')
df_confidence_intervals = logistic_reg_stats_models.conf_int()

# Combine them into a single summary DataFrame
odds_summary = pd.concat([df_coefficients, df_confidence_intervals], axis=1)
odds_summary.columns = ['Log-Odds', 'Lower CI (Log)', 'Upper CI (Log)']

# Exponentiate the entire DataFrame to calculate Odds Ratios
odds_summary['Odds Ratio'] = np.exp(odds_summary['Log-Odds'])
odds_summary['OR Lower CI'] = np.exp(odds_summary['Lower CI (Log)'])
odds_summary['OR Upper CI'] = np.exp(odds_summary['Upper CI (Log)'])

# Display the finalized odds ratios
odds_summary[['Odds Ratio', 'OR Lower CI', 'OR Upper CI']]

# Logistic Regression Model using Scikit-learn

In [ ]:
# Scale features (Highly recommended for faster solver convergence) on just the Training partition to prevent data leakage

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize and fit the Logistic Regression model

logistic_reg_sklearn_model = LogisticRegression(solver='lbfgs', max_iter=100)
logistic_reg_sklearn_model.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions

y_pred  = logistic_reg_sklearn_model.predict(X_test_scaled)              # Predict binary class (0 or 1)
y_probs = logistic_reg_sklearn_model.predict_proba(X_test_scaled)[:, 1]  # Predict raw probabilities

In [ ]:
# Evaluate the performance

print("Logsitic Regression Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.2f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nAUC score:")
print(roc_auc_score(y_test, y_probs))

# Machine Learning Model using Scikit-learn

In [ ]:
# Initialize and train the Gradient Boosting model on just the training partition

grad_boost_sklearn_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42, max_depth=3)
grad_boost_sklearn_model.fit(X_train, y_train)

In [ ]:
# Score and Evaluate on the hold-out validation partition

y_pred = grad_boost_sklearn_model.predict(X_test)                      # Predict binary class (0 or 1)
y_probs = grad_boost_sklearn_model.predict_proba(X_test_scaled)[:, 1]  # Predict raw probabilities

In [ ]:
# Evaluate the performance

print("Machine Learning Model Evaluation on Hold-out Data:")
print("-" * 40)
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nAUC score:")
print(roc_auc_score(y_test, y_probs))